# Trajectories, diversity, and corpus quality

A hands-on tour of the richer `pymc-generator`: **17 synthetic datasets**, each 104 weeks long.

| Dataset set | Worlds | Purpose |
| --- | ---: | --- |
| Mixed shard | 8 (4 generation cells × 2 draws) | Sparse worlds, component prevalence, active-count coverage, quality report |
| Trajectory atlas | 8 (one per archetype) | Recipes for launches, flights, ramps, shutdowns, doubling, seasonality, and trends |
| Replay world | 1 | Every component enabled; exact replay, reusable template, fixed-input oracle |

We first generate and **save** a shard, then analyze the file. You can replace the file path to inspect your own shard without regenerating it. Every plot and table runs in the docs build.

**Read the units correctly.** Model inputs are `treatment_raw` (nonnegative channel sizes), `covariates` (signed controls), and `outcome_raw`. Normalized training tensors are separate. Component flags, schedules, coefficients, and causal contributions are **privileged generator truth**, not features to give a fitted model.

This implements the sparse example in [#29](https://github.com/pymc-labs/pymc-generator/issues/29) and the stored-corpus report in [#31](https://github.com/pymc-labs/pymc-generator/issues/31), using features [#24–#28](https://github.com/pymc-labs/pymc-generator/pull/38). This tiny showcase is not a representative training corpus.


## 1. Install and choose seeds

Use **Python 3.13+** and the checkout containing this notebook (branched from PR #38). Older released packages can share the version number while lacking these unmerged features; the checkout revision and locked dependencies matter.

```bash
uv sync --locked --extra docs
.venv/bin/python -m ipykernel install --prefix "$PWD/.venv" --name pymc-generator
# Open this notebook in your Jupyter-capable editor with that kernel, or execute it:
.venv/bin/python -m nbconvert --execute --to html \
  --ExecutePreprocessor.timeout=600 --output-dir /tmp \
  docs/examples/trajectory-showcase.ipynb
```

No GPU, posterior sampling, or external data is needed. Set `PYMC_GENERATOR_SHOWCASE_DIR` before execution to keep outputs in a directory you choose; otherwise they go into a new temporary directory. The analysis/visual seed is independent of the generation seed.


In [ ]:
%matplotlib inline

import os
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc
import pytensor
from IPython.display import Image, display

import pymc_generator as pg
from pymc_generator import viz
from pymc_generator.presets import TRAJECTORY_ARCHETYPES
from pymc_generator.slots import TRAJECTORY_COMPONENTS, SlotLayout

GENERATION_SEED = 29
VISUAL_SEED = 31
VIF_THRESHOLD = 10.0
MAX_BAD_SHARE = 0.25

output_override = os.environ.get("PYMC_GENERATOR_SHOWCASE_DIR")
OUTPUT_DIR = (
    Path(output_override)
    if output_override
    else Path(tempfile.mkdtemp(prefix="pymc-generator-showcase-"))
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT_DIR))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print(
    "Versions:",
    {"pymc-generator": pg.__version__, "pymc": pymc.__version__, "pytensor": pytensor.__version__},
)
print("Kernel interpreter:", sys.executable)
print("Imported library:", pg.__file__)
print("Output directory:", RUN_DIR.resolve())

## 2. Generate a sparse, mixed-trajectory shard

Every world has **15 treatment slots and 10 control slots**, with 5–15 and 3–10 active inputs respectively. Activity masks distinguish padded slots from real zero-valued inputs. Exactly **five `C→Y` edges and three `Z→Y` edges** are present. All other edges are zero; the retained latent is disconnected.

We allocate one generation cell to each of four count combinations. All 88 combinations in the 11×8 grid remain visible in the report; 84 are deliberately given zero weight. To request uniform coverage of the whole grid, set `active_count_weights=None` and use at least 88 cells (outside this small tour).

`trajectories="composable"` gives each input independent inclusion flags for eight components: HF noise, pulses, onset, offset, flighting, level jumps, seasonal level, and trend. Flags are shared by draws from a cell; numerical schedules and series vary by draw. Realized prevalence need not equal the configured probability in four cells.

Mechanisms are **geometric carryover + Michaelis–Menten saturation**. The geometric kernel uses `pymc_marketing.geometric_adstock(normalize=True)`: weights sum to one, but inputs are **not** mapped to `[0, 1]`. We opt into log-uniform MM scales and reference-contribution priors; their anchors depend on parameters, never on the realized input mean.


In [ ]:
SPARSE_EDGES = dict.fromkeys(("cy", "dc", "dz", "dy", "zy", "zc", "cc", "zz"), (0, 0))
SPARSE_EDGES.update(cy=(5, 5), zy=(3, 3))
COUNT_COMBINATIONS = ((5, 3), (8, 5), (11, 7), (15, 10))
weights = np.zeros((11, 8))  # rows: 5..15 treatments; columns: 3..10 controls
for n_treatments, n_controls in COUNT_COMBINATIONS:
    weights[n_treatments - 5, n_controls - 3] = 1.0

shapes = pg.SCMPrior().saturation_prior_ranges
shapes["michaelis_menten"]["kappa_mult"] = (0.05, 20.0)
MECHANISMS = {
    "carryover_family_probs": {"none": 0.0, "geometric": 1.0, "weibull": 0.0},
    "saturation_family_probs": {
        name: float(name == "michaelis_menten")
        for name in ("linear", "hill", "logistic", "michaelis_menten", "tanh", "root")
    },
    "saturation_prior_ranges": shapes,
    "mm_scale_prior": "log_uniform",
    "treatment_reference_contribution_range": (0.5, 2.0),
    "covariate_reference_contribution_range": (-0.3, 0.3),
    "covariate_reference_scale": 2.0,
}
cfg = pg.make_scm_prior(
    n_treatments=15,
    n_covariates=10,
    n_latent=1,
    n_time_steps=104,
    n_treatments_active_range=(5, 15),
    n_covariates_active_range=(3, 10),
    n_cells=4,
    draws_per_cell=2,
    seed=GENERATION_SEED,
    edge_budget=SPARSE_EDGES,
    trajectories="composable",
    active_count_allocation="stratified",
    active_count_weights=weights.tolist(),
    **MECHANISMS,
)
generated = pg.sample_prior_predictive(cfg)
CORPUS_PATH = RUN_DIR / "mixed-worlds.npz"
pg.save_corpus(generated, CORPUS_PATH)
corpus = pg.load_corpus(CORPUS_PATH)
assert not pg.DataGenerator.validate_corpus(corpus)
for key, value in generated.items():
    if isinstance(value, np.ndarray):
        np.testing.assert_array_equal(corpus[key], value, err_msg=key)
    elif key == "identifiability":
        for name, array in value.items():
            np.testing.assert_array_equal(corpus[key][name], array, err_msg=name)
assert corpus["diagnostics"] == {
    key: value for key, value in generated["diagnostics"].items() if key != "timing"
}
print("Saved and loaded:", CORPUS_PATH.name, corpus["treatment_raw"].shape)
print(
    "Schema:",
    corpus["diagnostics"]["schema_version"],
    "| all numerical fields round-trip exactly; timing is not persisted",
)

In [ ]:
layout = SlotLayout(15, 10, 1)
edge_rows = []
for world_id in range(len(corpus["cell_id"])):
    graph = layout.unpack(corpus["g"][world_id])
    counts = {kind: int(block.sum()) for kind, block in graph.items()}
    assert counts == {kind: budget[0] for kind, budget in SPARSE_EDGES.items()}
    edge_rows.append(
        {
            "world_id": world_id,
            "cell_id": int(corpus["cell_id"][world_id]),
            "treatments": int(corpus["treatment_active_mask"][world_id].sum()),
            "controls": int(corpus["covariate_active_mask"][world_id].sum()),
            **counts,
        }
    )
assert corpus["treatment_raw"].shape == (8, 104, 15)
assert corpus["covariates"].shape == (8, 104, 10)
assert (corpus["treatment_raw"] >= 0).all()
assert (corpus["covariates"] < 0).any()  # signed controls are actually exercised
assert not corpus["latent_unobserved_contribution"].any()
assert not corpus["indirect_effects_by_source"].any()
for role, key in (("treatment", "treatment_raw"), ("covariate", "covariates")):
    active = corpus[f"{role}_active_mask"].astype(bool)
    assert not corpus[key].transpose(0, 2, 1)[~active].any()
    assert not corpus[key][corpus[f"{role}_activity"] == 0].any()
assert not corpus["treatment_shock_mask"].any()  # gates are the realized on-state in this example
assert (corpus["carryover_family"][corpus["treatment_active_mask"] == 1] == 1).all()
assert (corpus["sat_family"][corpus["treatment_active_mask"] == 1] == 3).all()
display(pd.DataFrame(edge_rows).set_index("world_id"))

### Four worlds: inputs and outcome decomposition

One world per generation cell, not four repeated draws of one structure. Every active input is shown. A channel can be observed yet have no direct effect: activity is not the same as a `C→Y` edge. The decomposition includes signed control effects and observation noise; these are not input trajectories.


In [ ]:
def plot_world(c, world_id):
    """Plot one stored world; retain original IDs and omit only padding."""
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, layout="constrained")
    for ax, role, key, prefix in (
        (axes[0], "treatment", "treatment_raw", "C"),
        (axes[1], "covariate", "covariates", "Z"),
    ):
        slots = np.flatnonzero(c[f"{role}_active_mask"][world_id])
        for slot in slots:
            ax.plot(c[key][world_id, :, slot], linewidth=1, label=f"{prefix}{slot + 1}")
        ax.set_ylabel(f"{role} input units")
        ax.legend(ncol=5, fontsize=7, loc="upper right")
    parts = {
        "intrinsic baseline": c["baseline_intrinsic"][world_id],
        "direct treatments": c["treatment_contribution_raw"][world_id].sum(axis=1),
        "controls": c["covariate_contribution"][world_id].sum(axis=1),
        "latents": c["latent_unobserved_contribution"][world_id].sum(axis=1),
        "indirect": c["indirect_effects_by_source"][world_id].sum(axis=1),
        "observation noise": c["outcome_noise"][world_id],
    }
    reconstructed = sum(np.asarray(value, dtype=np.float64) for value in parts.values())
    np.testing.assert_allclose(reconstructed, c["outcome_raw"][world_id], rtol=3e-6, atol=3e-6)
    for label, values in parts.items():
        axes[2].plot(values, label=label, linewidth=1)
    axes[2].plot(c["outcome_raw"][world_id], color="black", label="outcome", linewidth=1.5)
    axes[2].set(xlabel="reported week (zero-based)", ylabel="outcome units")
    axes[2].legend(ncol=4, fontsize=7)
    fig.suptitle(
        f"World {world_id} · cell {int(c['cell_id'][world_id])} · decomposition max error {np.max(np.abs(reconstructed - c['outcome_raw'][world_id])):.2g}"
    )
    plt.show()
    plt.close(fig)

In [ ]:
_, first_per_cell = np.unique(corpus["cell_id"], return_index=True)
for world_id in first_per_cell:
    plot_world(corpus, int(world_id))

## 3. Measure trajectories, not just component flags

Two distinct questions:

* **What was configured and drawn?** Stored flags and exact schedule leaves answer this. A seasonal flag can coexist with flighting and a negative trend.
* **What does the model actually receive?** Measure `treatment_raw` / `covariates`, including noise and gates, rather than the outcome contribution after carryover/saturation.

Below each active input gets zero fraction, first/last nonzero week, off→on/on→off transitions, and on-run count. Trend slopes use **nonzero weeks**, time scaled to `[0, 1]`, and input divided by its RMS over those weeks. Signed slope tracks the input value; magnitude slope tracks `abs(input)` (so a negative control approaching zero is decaying in magnitude). Fewer than two nonzero weeks yields an undefined slope, not zero.

The visible-shape screen calls a delayed input a **ramp after launch** if its magnitude slope is at least `0.25`, and a stopped input a **decay to shutdown** if it is at most `-0.25`. These are configurable descriptive heuristics, **not** monotonicity proofs, component classifiers, or statistical tests. An exact doubling *schedule* means a stored treatment jump factor equals two, not that two noisy neighboring observations have ratio two.


In [ ]:
def normalized_slopes(x, on):
    indexes = np.flatnonzero(on)
    if indexes.size < 2:
        return np.nan, np.nan
    values = np.asarray(x[indexes], dtype=np.float64)
    values = values / np.max(np.abs(values))
    values = values / np.sqrt(np.mean(values**2))
    time = indexes / max(len(x) - 1, 1)
    time = time - time.mean()
    denominator = time @ time
    return float(time @ (values - values.mean()) / denominator), float(
        time @ (np.abs(values) - np.abs(values).mean()) / denominator
    )


def measure_inputs(c, *, slope_threshold=0.25):
    records = []
    has_truth = "treatment_components" in c and "covariate_components" in c
    for role, key, prefix in (
        ("treatment", "treatment_raw", "C"),
        ("covariate", "covariates", "Z"),
    ):
        for world_id, slot in np.argwhere(c[f"{role}_active_mask"] == 1):
            x = c[key][world_id, :, slot]
            on = x != 0.0
            indexes = np.flatnonzero(on)
            signed_slope, magnitude_slope = normalized_slopes(x, on)
            launches = int(np.count_nonzero(~on[:-1] & on[1:]))
            stops = int(np.count_nonzero(on[:-1] & ~on[1:]))
            row = {
                "world_id": int(world_id),
                "role": role,
                "slot": int(slot),
                "input": f"{prefix}{slot + 1}",
                "zero_fraction": float((~on).mean()),
                "first_nonzero_week": int(indexes[0]) if indexes.size else None,
                "last_nonzero_week": int(indexes[-1]) if indexes.size else None,
                "off_to_on": launches,
                "on_to_off": stops,
                "on_runs": launches + int(on[0]),
                "signed_slope": signed_slope,
                "magnitude_slope": magnitude_slope,
            }
            row.update(
                always_on=bool(on.all()),
                all_zero=bool(not on.any()),
                delayed_start=bool(indexes.size and indexes[0] > 0),
                terminal_shutdown=bool(indexes.size and indexes[-1] < len(x) - 1),
                off_on_again=bool(row["on_runs"] >= 2),
            )
            row["ramp_after_launch"] = row["delayed_start"] and magnitude_slope >= slope_threshold
            row["decay_to_shutdown"] = (
                row["terminal_shutdown"] and magnitude_slope <= -slope_threshold
            )
            if has_truth:
                flags = c[f"{role}_components"][world_id, slot].astype(bool)
                row.update(
                    {
                        f"carries_{name}": bool(flag)
                        for name, flag in zip(TRAJECTORY_COMPONENTS, flags)
                    }
                )
                row["schedule_trend_change"] = (
                    float(c[f"trajectory_{role}_trend_change"][world_id, slot])
                    if row["carries_trend"]
                    else 0.0
                )
                row["seasonal_period_weeks"] = (
                    float(c[f"trajectory_{role}_seasonal_period"][world_id, slot])
                    if row["carries_seasonal"]
                    else np.nan
                )
                row["exact_doubling_schedule"] = bool(
                    role == "treatment"
                    and row["carries_level_jump"]
                    and (
                        c["trajectory_treatment_level_jump_factor"][world_id, :, slot] == 2.0
                    ).any()
                )
            records.append(row)
    return pd.DataFrame.from_records(records)


OBSERVED_SHAPES = (
    "always_on",
    "all_zero",
    "delayed_start",
    "terminal_shutdown",
    "off_on_again",
    "ramp_after_launch",
    "decay_to_shutdown",
)


def trajectory_inventory(measurements):
    rows = []
    for role in ("treatment", "covariate"):
        group = measurements[measurements["role"] == role]
        for name in OBSERVED_SHAPES:
            count = int(group[name].sum())
            rows.append(
                {
                    "role": role,
                    "trajectory": name,
                    "evidence": "reported input",
                    "count": count,
                    "eligible_inputs": len(group),
                    "status": "observed" if count else "absent in this shard",
                }
            )
        if role == "treatment" and "exact_doubling_schedule" in group:
            count = int(group["exact_doubling_schedule"].sum())
            rows.append(
                {
                    "role": role,
                    "trajectory": "exact_doubling_schedule",
                    "evidence": "stored schedule truth",
                    "count": count,
                    "eligible_inputs": len(group),
                    "status": "observed" if count else "absent in this shard",
                }
            )
    return pd.DataFrame(rows)

In [ ]:
# Boundary examples make the observable measurement definitions inspectable.
boundary_cases = {
    "all_zero": [0, 0, 0, 0, 0],
    "first_week_only": [2, 0, 0, 0, 0],
    "last_week_only": [0, 0, 0, 0, 2],
    "negative_constant": [-2, -2, -2, -2, -2],
    "zero_signed_mean": [-1, 0, 1],
    "approaching_zero": [-3, -2, -1],
    "gapped_rise": [0, 1, 0, 0, 4],
    "three_flights": [1, 0, 1, 0, 1],
}
boundary_rows = []
for label, values in boundary_cases.items():
    x = np.asarray(values, dtype=np.float64)
    view = {
        "treatment_raw": np.empty((1, len(x), 0)),
        "treatment_active_mask": np.empty((1, 0), dtype=np.uint8),
        "covariates": x[None, :, None],
        "covariate_active_mask": np.ones((1, 1), dtype=np.uint8),
    }
    row = measure_inputs(view).iloc[0]
    boundary_rows.append(row.to_dict() | {"example": label})
boundaries = pd.DataFrame(boundary_rows).set_index("example")
assert boundaries.loc["all_zero", "on_runs"] == 0
assert pd.isna(boundaries.loc["all_zero", "first_nonzero_week"])
assert boundaries.loc["first_week_only", "off_to_on"] == 0
assert boundaries.loc["first_week_only", "on_to_off"] == 1
assert boundaries.loc["last_week_only", "off_to_on"] == 1
assert boundaries.loc["last_week_only", "on_to_off"] == 0
assert boundaries.loc["negative_constant", "on_runs"] == 1
np.testing.assert_array_equal(
    boundaries.loc["negative_constant", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    0.0,
)
assert (
    boundaries.loc[["all_zero", "first_week_only", "last_week_only"], "signed_slope"].isna().all()
)
np.testing.assert_allclose(
    boundaries.loc["zero_signed_mean", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    [2.0, 0.0],
)
np.testing.assert_allclose(
    boundaries.loc["approaching_zero", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    [2 / np.sqrt(14 / 3), -2 / np.sqrt(14 / 3)],
)
np.testing.assert_allclose(
    boundaries.loc["gapped_rise", ["signed_slope", "magnitude_slope"]].to_numpy(dtype=float),
    4 / np.sqrt(8.5),
)
assert boundaries.loc["three_flights", "on_runs"] == 3
assert boundaries.loc["three_flights", "off_to_on"] == 2
assert boundaries.loc["three_flights", "on_to_off"] == 2
display(
    boundaries[
        [
            "first_nonzero_week",
            "last_nonzero_week",
            "off_to_on",
            "on_to_off",
            "on_runs",
            "signed_slope",
            "magnitude_slope",
        ]
    ].round(3)
)

In [ ]:
measurements = measure_inputs(corpus)
display(
    measurements[
        [
            "world_id",
            "role",
            "input",
            "zero_fraction",
            "off_to_on",
            "on_to_off",
            "on_runs",
            "signed_slope",
            "magnitude_slope",
            "schedule_trend_change",
        ]
    ]
    .head(16)
    .round(3)
)
display(trajectory_inventory(measurements))

## 4. A deterministic quality report from the file

`quality_report(path, ...)` below does **no generation** and does not depend on `cfg`. It reloads the archive, validates it, recalculates prevalence and coverage from masks, computes descriptors and within-world diagnostics, and selects seeded visual examples.

**Prevalence denominator:** active `(world, input)` pairs, never padded slots or off-weeks. This matches the stored diagnostics. Eight worlds represent **four independent generation structures**, so repeated flags across a cell's draws are not eight independent component draws.

**Empirical identifiability screen:** observed active `C + Z` design, centered and unit-column-normalized by the library. A world is flagged if its numerical rank is below the active predictor count **or** any valid predictor VIF is at least `VIF_THRESHOLD` (infinite VIF counts). Constants reduce rank; invalid or absent designs are explicitly unassessed, never automatic passes. Read **differences first**, with levels as a companion: independent random walks can have high level VIF. Condition numbers are also reported, but have no extra threshold here.

The rank uses the library's SVD tolerance `max(eps64, eps64 × max(n_observations, n_varying_columns) × largest_singular_value)`. VIF is computed by residual projection onto the other predictors, not inferred from rank. The default threshold `10` is a notebook policy, **not a library default**. The acceptable flagged share defaults to `25%` and must remain below `50%` if the policy is to require a minority.

This is a finite-sample redundancy screen of raw inputs, **not proof of causal identifiability, nonlinear-response parameter recovery, or training utility**. Quality PASS/FAIL is reported honestly and never filters worlds away.

For **report-only use**, set `PYMC_GENERATOR_REPORT_PATH` to an existing shard and execute all cells tagged `report-setup`, `report-definitions`, or `report-run`, in notebook order. Skip generation and atlas cells; the tagged path includes the shared plotting helpers and seeded input/world visual checks. The [corpus-quality guide](https://pymc-labs.github.io/pymc-generator/guide/corpus-quality/) also gives a fresh-kernel command. PASS means a completely assessed, nonempty selection with flagged share **≤** the printed limit.


In [ ]:
DESCRIPTOR_FEATURES = (
    "treatment_cv_median",
    "treatment_zero_fraction_median",
    "treatment_roughness_median",
    "treatment_acf_lag_1_median",
    "covariate_level_to_variation_median",
    "treatment_pair_diff_abs_pearson_max",
    "outcome_cv",
)
DESCRIPTOR_LAGS = (1, 4, 13, 26, 52)


def component_prevalence(c):
    block = c["diagnostics"].get("trajectory")
    if block is None:
        return pd.DataFrame(
            columns=[
                "role",
                "component",
                "carried_inputs",
                "active_inputs",
                "share",
                "configured_probability",
            ]
        )
    assert tuple(block["components"]) == TRAJECTORY_COMPONENTS
    rows = []
    for role in ("treatment", "covariate"):
        active = c[f"{role}_active_mask"] == 1
        denominator = int(active.sum())
        counts = c[f"{role}_components"][active].sum(axis=0)
        assert block["n_inputs"][role] == denominator
        for component, count in zip(TRAJECTORY_COMPONENTS, counts):
            stored_share = int(count) / denominator if denominator else 0.0
            assert block["prevalence"][role][component] == stored_share
            rows.append(
                {
                    "role": role,
                    "component": component,
                    "carried_inputs": int(count),
                    "active_inputs": denominator,
                    "share": stored_share if denominator else np.nan,
                    "configured_probability": block["inclusion_probs"][role][component],
                }
            )
    return pd.DataFrame(rows)


def mask_coverage(c):
    block = c["diagnostics"].get("active_count_coverage")
    treatments = c["treatment_active_mask"].sum(axis=1).astype(int)
    controls = c["covariate_active_mask"].sum(axis=1).astype(int)
    if block is not None:
        coverage = pg.active_count_coverage(c)
        for key in ("n_treatments_active", "n_covariates_active", "n_worlds", "n_cells"):
            assert coverage[key] == block[key]
    else:
        # Without allocation metadata, only observed bounds are known from a file.
        treatment_axis = list(range(int(treatments.min()), int(treatments.max()) + 1))
        control_axis = list(range(int(controls.min()), int(controls.max()) + 1))
        coverage = {"n_treatments_active": treatment_axis, "n_covariates_active": control_axis}
    world_counts = np.zeros(
        (len(coverage["n_treatments_active"]), len(coverage["n_covariates_active"])), dtype=int
    )
    cell_counts = np.zeros_like(world_counts)
    for i, n_treatments in enumerate(coverage["n_treatments_active"]):
        for j, n_controls in enumerate(coverage["n_covariates_active"]):
            selected = (treatments == n_treatments) & (controls == n_controls)
            world_counts[i, j] = int(selected.sum())
            cell_counts[i, j] = len(np.unique(c["cell_id"][selected]))
    if block is not None:
        np.testing.assert_array_equal(world_counts, coverage["n_worlds"])
        np.testing.assert_array_equal(cell_counts, coverage["n_cells"])
    coverage.update(n_worlds=world_counts.tolist(), n_cells=cell_counts.tolist())
    return coverage


def design_screen(diagnostics, *, vif_threshold):
    rows = []
    for view in ("levels", "differences"):
        result = diagnostics[view].vif["observed"]
        for position, world_id in enumerate(result.world_ids):
            n_predictors = int(result.eligible[position].sum())
            valid_values = result.vif[position, result.valid[position]]
            varying = result.eligible[position] & ~result.constant[position]
            assessed = (
                n_predictors > 0
                and result.n_time_steps >= 3
                and bool(result.valid[position, varying].all())
            )
            maximum = float(valid_values.max()) if valid_values.size else np.nan
            deficient = int(result.rank[position]) < n_predictors
            near_collinear = bool(valid_values.size and (valid_values >= vif_threshold).any())
            rows.append(
                {
                    "world_id": int(world_id),
                    "view": view,
                    "active_predictors": n_predictors,
                    "rank": int(result.rank[position]),
                    "constant_predictors": int(result.constant[position].sum()),
                    "condition": float(result.condition[position]),
                    "max_vif": maximum,
                    "assessed": assessed,
                    "rank_deficient": deficient,
                    "near_collinear": near_collinear,
                    "flagged": deficient or near_collinear,
                }
            )
    return pd.DataFrame(rows)


def quality_report(
    path, *, seed=31, vif_threshold=10.0, max_bad_share=0.25, design_view="differences"
):
    if not np.isfinite(vif_threshold) or vif_threshold < 1:
        raise ValueError("vif_threshold must be finite and >= 1")
    if not 0 <= max_bad_share < 0.5:
        raise ValueError("max_bad_share must be in [0, 0.5) to require a minority")
    if design_view not in ("levels", "differences"):
        raise ValueError("design_view must be levels or differences")
    c = pg.load_corpus(path)
    errors = pg.DataGenerator.validate_corpus(c)
    if errors:
        raise ValueError("Invalid corpus: " + "; ".join(errors))
    rows = pg.world_descriptors(c, source_id="stored_shard", lags=DESCRIPTOR_LAGS)
    diagnostics = pg.data_diagnostics(
        c, scopes=("nodes", "decomposition"), lags=(1, 4, 13, 26, 52), keep_series=False
    )
    design = design_screen(diagnostics, vif_threshold=vif_threshold)
    screen = design[design["view"] == design_view]
    unassessed = int((~screen["assessed"]).sum())
    bad_share = float(screen["flagged"].mean()) if not unassessed else np.nan
    decision = {
        "view": design_view,
        "vif_threshold": float(vif_threshold),
        "max_bad_share": float(max_bad_share),
        "n_worlds": len(screen),
        "n_unassessed": unassessed,
        "rank_deficient_share": float(screen["rank_deficient"].mean()),
        "near_collinear_share": float(screen["near_collinear"].mean()),
        "flagged_share": bad_share,
        "status": "UNASSESSED" if unassessed else "PASS" if bad_share <= max_bad_share else "FAIL",
    }
    trajectories = measure_inputs(c)
    rng = np.random.default_rng(seed)
    random_worlds = np.sort(
        rng.choice(len(c["cell_id"]), size=min(3, len(c["cell_id"])), replace=False)
    )
    candidates = trajectories[trajectories["world_id"].isin(random_worlds)]
    positions = rng.choice(len(candidates), size=min(6, len(candidates)), replace=False)
    random_inputs = candidates.iloc[positions][["world_id", "role", "slot", "input"]].reset_index(
        drop=True
    )
    return {
        "corpus": c,
        "descriptors": rows,
        "summary": pg.summarize_descriptors(rows, features=DESCRIPTOR_FEATURES),
        "diagnostics": diagnostics,
        "prevalence": component_prevalence(c),
        "coverage": mask_coverage(c),
        "design": design,
        "decision": decision,
        "trajectories": trajectories,
        "inventory": trajectory_inventory(trajectories),
        "random_worlds": random_worlds,
        "random_inputs": random_inputs,
    }

In [ ]:
# Report from file: set PYMC_GENERATOR_REPORT_PATH for report-only execution.
report_override = os.environ.get("PYMC_GENERATOR_REPORT_PATH")
REPORT_PATH = Path(report_override) if report_override else CORPUS_PATH
report = quality_report(
    REPORT_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
)
repeat = quality_report(
    REPORT_PATH, seed=VISUAL_SEED, vif_threshold=VIF_THRESHOLD, max_bad_share=MAX_BAD_SHARE
)
for key in ("prevalence", "design", "trajectories", "inventory", "random_inputs"):
    pd.testing.assert_frame_equal(report[key], repeat[key], check_exact=True)
pd.testing.assert_frame_equal(
    report["descriptors"].to_frame(with_status=True),
    repeat["descriptors"].to_frame(with_status=True),
    check_exact=True,
)
pd.testing.assert_frame_equal(
    report["summary"].to_frame(), repeat["summary"].to_frame(), check_exact=True
)
np.testing.assert_array_equal(report["random_worlds"], repeat["random_worlds"])
assert report["coverage"] == repeat["coverage"]
assert report["decision"] == repeat["decision"]
print("Report reproduced exactly from the stored file at seed", VISUAL_SEED)
print("Design screen:", report["decision"])
display(report["design"].round(3))
display(report["inventory"])

In [ ]:
if report["prevalence"].empty:
    print("Trajectory component truth unavailable in this archive; no prevalence inferred.")
else:
    display(report["prevalence"].round(3))
    prevalence = (
        report["prevalence"]
        .pivot(index="component", columns="role", values="share")
        .reindex(TRAJECTORY_COMPONENTS)
    )
    ax = prevalence.plot.bar(
        figsize=(10, 3),
        ylim=(0, 1),
        rot=0,
        ylabel="share of active (world, input) pairs",
        title="Carried components — not a classifier of visible shapes",
    )
    plt.show()
    plt.close(ax.figure)

coverage = report["coverage"]
allocation = report["corpus"]["diagnostics"].get("active_count_coverage")
weight_values = allocation.get("weights") if allocation is not None else None
stored_weights = np.asarray(weight_values) if weight_values is not None else None
fig, axes = plt.subplots(1, 2, figsize=(11, 6), layout="constrained")
for ax, name in zip(axes, ("n_cells", "n_worlds")):
    counts = np.asarray(coverage[name])
    excluded = (
        stored_weights == 0 if stored_weights is not None else np.zeros_like(counts, dtype=bool)
    )
    cmap = plt.colormaps["Blues"].copy()
    cmap.set_bad("#eeeeee")
    image = ax.imshow(
        np.ma.masked_where(excluded, counts),
        cmap=cmap,
        vmin=0,
        vmax=max(1, counts.max()),
    )
    for i, j in np.ndindex(counts.shape):
        ax.text(
            j,
            i,
            "×" if excluded[i, j] else str(counts[i, j]),
            ha="center",
            va="center",
            fontsize=8,
        )
    ax.set(
        xticks=range(len(coverage["n_covariates_active"])),
        xticklabels=coverage["n_covariates_active"],
        yticks=range(len(coverage["n_treatments_active"])),
        yticklabels=coverage["n_treatments_active"],
        xlabel="active controls",
        ylabel="active treatments",
        title=name,
    )
    fig.colorbar(image, ax=ax, shrink=0.65)
plt.show()
plt.close(fig)
occupied = int((np.asarray(coverage["n_cells"]) > 0).sum())
if stored_weights is not None:
    print("× = zero allocation weight, not an accidental coverage gap.")
    print("Positive-weight combinations reached:", occupied, "/", int((stored_weights > 0).sum()))
if allocation is None:
    print(
        "Allocation metadata unavailable; axes cover observed count bounds, not the intended grid."
    )
else:
    print("Full configured grid reached:", occupied, "/", np.asarray(coverage["n_worlds"]).size)

### Descriptor distributions and within-world diagnostics

A descriptor row summarizes **one world** over active inputs. The summary retains finite, infinite, undefined, and ineligible counts. Histograms below show finite values only, with excluded-value counts in the title; the table is the authoritative status-aware report.

Read shape, co-movement, contribution size, and rank together. A schema-valid world can still have little useful treatment signal. The signal gate is a separate truth-based diagnostic, not the raw-input VIF screen.


In [ ]:
display(report["summary"].to_frame().round(3))
features_to_plot = DESCRIPTOR_FEATURES[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 6), layout="constrained")
for ax, feature in zip(axes.flat, features_to_plot):
    values = report["descriptors"].column(feature)
    finite = np.isfinite(values)
    ax.hist(values[finite], bins=5, color="#306998", edgecolor="white")
    ax.set(title=f"{feature}\nfinite {finite.sum()}/{len(values)}", ylabel="worlds")
    ax.tick_params(labelsize=8)
    ax.title.set_fontsize(8)
plt.show()
plt.close(fig)
print(
    pg.summarize_descriptors(
        report["descriptors"],
        features=("treatment_cv_median",),
        by=("n_treatments_active", "n_covariates_active"),
    ).table("treatment_cv_median")
)
print(report["diagnostics"].contributions.table())
print(pg.outcome_distributions(report["corpus"]).table(of="share"))

In [ ]:
def show_saved_plot(plot_fn, source, filename, **kwargs):
    path = RUN_DIR / filename
    plot_fn(source, str(path), **kwargs)
    display(Image(filename=str(path)))


show_saved_plot(viz.plot_vif_diagnostics, report["diagnostics"], "mixed-vif.png")
show_saved_plot(viz.plot_contribution_diagnostics, report["diagnostics"], "mixed-contributions.png")
from pymc_generator.signal_diagnostics import check_signal_gate

signal = report["corpus"]["diagnostics"].get("signal")
if signal is None:
    print("Stored signal diagnostics unavailable in this archive.")
else:
    print("Stored signal summary:", signal)
    print("Default signal gate:", check_signal_gate(signal))

## 5. Seeded visual checks and manual selection

The report's seed selects worlds **and** input series without replacement. Each row below names its original world ID, role, and slot. For treatments, the schedule envelope is a multiplicative `exp(log_level_shift)`; for controls, it is an additive shift in control units. A gate is separate from the envelope. A treatment shock could override an off-week; this shard has no shocks.

`MANUAL_WORLD_IDS` and `MANUAL_INPUTS` are the knobs for curation. They select **existing** datasets, never generate replacements. We translate the explicit ID list to a boolean `worlds=` mask, preserving original IDs and avoiding integer-mask ambiguity for a two-world shard. Diagnostic tables use archive order; the world plots follow your list order. An empty input list still permits world-only inspection.


In [ ]:
def plot_input_selection(c, selection):
    if selection.empty:
        print("No individual inputs selected; world plots remain available.")
        return
    fig, axes = plt.subplots(
        len(selection),
        2,
        figsize=(12, max(3, 2.1 * len(selection))),
        squeeze=False,
        layout="constrained",
    )
    for row, (world_id, role, slot, label) in enumerate(
        selection.itertuples(index=False, name=None)
    ):
        key = "treatment_raw" if role == "treatment" else "covariates"
        x = c[key][world_id, :, slot]
        axes[row, 0].plot(x, linewidth=1, label=f"world {world_id} · {label}")
        axes[row, 0].set_ylabel("input units")
        axes[row, 0].legend(fontsize=8)
        shift_key = "treatment_log_level_shift" if role == "treatment" else "covariate_level_shift"
        if f"{role}_activity" in c and shift_key in c:
            gate = c[f"{role}_activity"][world_id, :, slot]
            axes[row, 0].fill_between(
                np.arange(len(x)),
                0,
                1,
                where=gate == 0,
                color="grey",
                alpha=0.2,
                transform=axes[row, 0].get_xaxis_transform(),
                label="gate off",
            )
            axes[row, 0].legend(fontsize=8)
            shift = c[shift_key][world_id, :, slot]
            envelope = np.exp(shift) if role == "treatment" else shift
            axes[row, 1].plot(
                envelope,
                color="#9c3e74",
                label="level multiplier" if role == "treatment" else "additive level shift",
            )
            axes[row, 1].legend(fontsize=8)
        else:
            axes[row, 1].text(
                0.5,
                0.5,
                "Schedule truth unavailable in this archive",
                ha="center",
                transform=axes[row, 1].transAxes,
            )
    for ax in axes[-1]:
        ax.set_xlabel("reported week")
    plt.show()
    plt.close(fig)


print("Seeded world IDs:", report["random_worlds"].tolist())
display(report["random_inputs"])
plot_input_selection(report["corpus"], report["random_inputs"])
for world_id in report["random_worlds"]:
    plot_world(report["corpus"], int(world_id))

In [ ]:
MANUAL_WORLD_IDS = [0, 6]  # edit this list; IDs refer to rows in REPORT_PATH
MANUAL_INPUTS = [(0, "treatment", 0), (6, "covariate", 0)]  # zero-based slots, C1 / Z1

manual_ids = np.asarray(MANUAL_WORLD_IDS)
n_available = len(report["corpus"]["cell_id"])
if manual_ids.ndim != 1 or manual_ids.dtype.kind not in "iu" or not manual_ids.size:
    raise ValueError("Select a nonempty list of integer world IDs")
if (
    (manual_ids < 0).any()
    or (manual_ids >= n_available).any()
    or len(np.unique(manual_ids)) != len(manual_ids)
):
    raise ValueError("World IDs must be unique positions within the stored shard")
manual_mask = np.isin(np.arange(n_available), manual_ids)
manual_descriptors = pg.world_descriptors(
    report["corpus"], source_id="stored_shard", worlds=manual_mask, lags=DESCRIPTOR_LAGS
)
manual_diagnostics = pg.data_diagnostics(
    report["corpus"], worlds=manual_mask, scopes=("nodes", "decomposition"), lags=(1, 4)
)
print(manual_descriptors.table(DESCRIPTOR_FEATURES))
print(manual_diagnostics.contributions.table())
manual_rows = []
for world_id, role, slot in MANUAL_INPUTS:
    if world_id not in MANUAL_WORLD_IDS:
        raise ValueError("Manual inputs must belong to the selected worlds")
    if role not in ("treatment", "covariate"):
        raise ValueError("Input role must be treatment or covariate")
    mask = report["corpus"][f"{role}_active_mask"][world_id]
    if not 0 <= slot < len(mask) or not mask[slot]:
        raise ValueError("Choose an active slot, not zero padding")
    manual_rows.append((world_id, role, slot, f"{'C' if role == 'treatment' else 'Z'}{slot + 1}"))
manual_selection = pd.DataFrame(manual_rows, columns=["world_id", "role", "slot", "input"])
plot_input_selection(report["corpus"], manual_selection)
for world_id in MANUAL_WORLD_IDS:
    plot_world(report["corpus"], world_id)

## 6. Generate the trajectory atlas: eight recipes

One small `SCM` per archetype, with two treatments, one signed control, direct effects only, and a disconnected latent. Unlike a training shard, a single world has no cell-level train/validation split; we export it as a **CSV/text/plot audit bundle**, not a fictitious one-world corpus.

The measurement adapter below is just a view of genuine `SCM.data` and `SCM.params["trajectory"]`, **not** a corpus that can be passed to `save_corpus`. It preserves the same raw-input measurement rules. The texture-only `always_on_spikes` world has no schedule block; we leave its component truth unavailable rather than inventing flags.

Presets configure schedules, not deterministic smooth curves. Noise remains visible. A pure treatment ramp changes its **log level**; control trends are additive and can have either sign. The doubling recipe fixes the treatment jump factor to exactly two and uses quiet, parentless controls whose additive jump equals their pinned level.


In [ ]:
ATLAS_EDGES = dict.fromkeys(SPARSE_EDGES, (0, 0))
ATLAS_EDGES.update(cy=(2, 2), zy=(1, 1))
atlas = {}
atlas_paths = {}
for index, name in enumerate(TRAJECTORY_ARCHETYPES):
    prior = pg.make_scm_prior(
        n_treatments=2,
        n_covariates=1,
        n_latent=1,
        n_time_steps=104,
        trajectories=name,
        edge_budget=ATLAS_EDGES,
        **MECHANISMS,
    )
    world = pg.sample_scm(prior, seed=100 + index, name=name)
    assert world is not None
    assert world.identity_error() < 1e-9
    atlas[name] = world
    atlas_paths[name] = pg.write_scm_bundle(world, str(RUN_DIR / name))
print("Atlas bundles:", list(atlas))
print("Each folder has model-facing dataset.csv and separate true_components.csv.")


def scm_measurement_view(world):
    view = {
        "treatment_raw": world.data["treatments"][None],
        "covariates": world.data["covariates"][None],
        "treatment_active_mask": np.ones((1, world.n_treatments), dtype=np.uint8),
        "covariate_active_mask": np.ones((1, world.n_covariates), dtype=np.uint8),
    }
    truth = world.params.get("trajectory")
    if truth is not None:
        for role in ("treatment", "covariate"):
            spec = truth[role]
            view[f"{role}_components"] = np.stack(
                [spec["use"][component] for component in TRAJECTORY_COMPONENTS], axis=-1
            )[None]
            for component, leaves in spec.items():
                if component != "use":
                    for leaf, values in leaves.items():
                        if leaf != "count":
                            view[f"trajectory_{role}_{component}_{leaf}"] = np.asarray(values)[None]
    return view


atlas_measurements = pd.concat(
    [
        measure_inputs(scm_measurement_view(world)).assign(archetype=name)
        for name, world in atlas.items()
    ],
    ignore_index=True,
)
display(
    atlas_measurements[
        [
            "archetype",
            "role",
            "input",
            "zero_fraction",
            "first_nonzero_week",
            "last_nonzero_week",
            "on_runs",
            "magnitude_slope",
            "schedule_trend_change",
            "seasonal_period_weeks",
            "exact_doubling_schedule",
        ]
    ].round(3)
)
for name, world in atlas.items():
    fig, axes = plt.subplots(1, 3, figsize=(12, 2.5), layout="constrained")
    for slot in range(world.n_treatments):
        axes[0].plot(world.data["treatments"][:, slot], label=f"C{slot + 1}", linewidth=1)
    axes[0].set(title=f"{name}: treatments", ylabel="input units")
    axes[0].legend(fontsize=7)
    axes[1].plot(world.data["covariates"][:, 0], color="#9c3e74", linewidth=1, label="Z1")
    axes[1].set(title="signed control", ylabel="input units")
    if "treatment_activity" in world.data:
        axes[2].plot(
            np.exp(world.data["treatment_log_level_shift"][:, 0]),
            label="C1 level multiplier",
            color="#b77c24",
        )
        axes[2].plot(
            world.data["treatment_activity"][:, 0], label="C1 gate", color="grey", alpha=0.7
        )
    else:
        axes[2].text(
            0.5,
            0.5,
            "Texture-only world:\nno stored schedule block",
            transform=axes[2].transAxes,
            ha="center",
        )
    axes[2].set(title="schedule truth (not raw input)")
    if "treatment_activity" in world.data:
        axes[2].legend(fontsize=7)
    for ax in axes:
        ax.set_xlabel("reported week")
    plt.show()
    plt.close(fig)

### Fill a measured gap deliberately

An absent row means **not observed in this finite shard**, not impossible under the library. For example, a continuously drawn jump factor will not reliably hit *exactly* two. The atlas's `level_doubling` recipe pins it to two. Likewise, `ramp_up` combines onset with positive trend; `decay_to_zero` combines negative trend with an exact offset gate. Exponential decay alone approaches zero; the gate supplies exact zeros.


In [ ]:
doubling = atlas["level_doubling"].params["trajectory"]["treatment"]["level_jump"]["factor"]
np.testing.assert_array_equal(doubling, 2.0)
assert not measurements.loc[measurements["role"] == "treatment", "exact_doubling_schedule"].any()
assert atlas_measurements.loc[
    atlas_measurements["archetype"] == "level_doubling", "exact_doubling_schedule"
].any()
print("Mixed shard: no exact x2 treatment schedule. Atlas: fixed x2 schedule demonstrated.")
for name in ("periodic_on_off", "delayed_start", "ramp_up", "decay_to_zero"):
    world = atlas[name]
    for role, key in (("treatment", "treatments"), ("covariate", "covariates")):
        assert not world.data[key][world.data[f"{role}_activity"] == 0].any()
print("All gated atlas inputs are exact zero in their off-weeks.")

## 7. Diversity between datasets: bins, neighbors, and comparisons

The APIs work on a corpus or a list of `SCM` objects. We keep source IDs distinct so an atlas world and a shard world cannot accidentally share an identity. Known generation-cell IDs let nearest-neighbor search exclude sibling draws.

Distances use a scale fitted on the **mixed reference**, not on the query. The chosen features are observable and nonconstant in this reference; adding a constant or nonfinite feature requires an explicit decision, not silently dropping it. Similar descriptors do not establish duplicate datasets, causal equivalence, or equal usefulness for training.


In [ ]:
atlas_rows = pg.WorldDescriptors.concat(
    [
        pg.world_descriptors([world], source_id=name, lags=DESCRIPTOR_LAGS)
        for name, world in atlas.items()
    ]
)
all_rows = pg.WorldDescriptors.concat([report["descriptors"], atlas_rows])
print(
    pg.summarize_descriptors(
        all_rows,
        features=("treatment_cv_median", "treatment_zero_fraction_median"),
        by=("source_id",),
    ).table("treatment_zero_fraction_median")
)
print(
    pg.bin_counts(
        report["descriptors"],
        {
            "treatment_cv_median": [0, 0.25, 0.5, 1, 2, 5],
            "treatment_zero_fraction_median": [0, 0.1, 0.3, 0.6, 1],
        },
    ).table()
)
SIMILARITY_FEATURES = (
    "treatment_cv_median",
    "treatment_roughness_median",
    "treatment_acf_lag_1_median",
)
within = pg.nearest_worlds(
    report["descriptors"], features=SIMILARITY_FEATURES, exclude_same_group=True
)
print("Nearest world from a different generation cell:")
print(within.table())
print("Atlas worlds relative to the mixed reference, on the same scale:")
print(
    pg.nearest_worlds(
        report["descriptors"], atlas_rows, features=SIMILARITY_FEATURES, scale=within.scale
    ).table()
)
print(
    pg.compare_descriptors(report["descriptors"], atlas_rows, features=SIMILARITY_FEATURES).table(
        "treatment_cv_median"
    )
)

## 8. Mechanism priors, exact replay, template parity, and an oracle

The ninth small world puts **all eight components on every input**. This is a capability check, not a prevalence sample. We keep the same geometric + MM response and parameter-only reference priors.

Reference targets are **nominal**, before edge gates. For MM at reference multiplier `m`, the response is `m / (m + kappa_mult)` and `beta × response = target`. A control uses `rho_zy × reference_input = target`. Neither promises the mean contribution over a noisy realized trajectory. In these unfloored worlds a control's actual effect remains the literal `(g_zy × rho_zy) × Z` on reported controls; saturation affects treatments, not controls.

Legacy corpus features are stored in float32; exact mechanism and continuous schedule leaves are float64. A single `SCM` retains its float64 forward values. We therefore use storage-scale tolerances for corpus reconstruction, **bitwise replay** in the same numerical environment, and tight numerical tolerances for separately compiled model paths.


In [ ]:
all_components = {
    f"{role}_{component}_inclusion_prob": 1.0
    for role in ("treatment", "covariate")
    for component in TRAJECTORY_COMPONENTS
}
replay_edges = dict.fromkeys(SPARSE_EDGES, (0, 0))
replay_edges.update(cy=(3, 3), zy=(2, 2))
replay_cfg = pg.make_scm_prior(
    n_treatments=3,
    n_covariates=2,
    n_latent=1,
    n_time_steps=104,
    trajectories="composable",
    edge_budget=replay_edges,
    **MECHANISMS,
    **all_components,
)
rich = pg.sample_scm(replay_cfg, seed=27, name="all_components")
assert rich is not None
for role in ("treatment", "covariate"):
    assert all(np.asarray(flag).all() for flag in rich.params["trajectory"][role]["use"].values())
replayed = rich.replay()  # executes the generative graph at the recorded accepted primitives
assert set(replayed) == set(rich.data)
for name, expected in rich.data.items():
    np.testing.assert_array_equal(replayed[name], expected, err_msg=name)
print(
    "Exact replay:",
    len(replayed),
    "forward outputs from",
    len(rich.primitive_parameters),
    "recorded primitive draws",
)

p = rich.params
multiplier = rich.cfg.treatment_reference_multiplier
np.testing.assert_allclose(
    p["beta"] * multiplier / (multiplier + p["mm_kappa_mult"]),
    p["treatment_reference_contribution"],
    rtol=3e-14,
    atol=0.0,
)
np.testing.assert_allclose(
    p["rho_zy"] * rich.cfg.covariate_reference_scale,
    p["covariate_reference_contribution"],
    rtol=3e-14,
    atol=0.0,
)
expected_control = (rich.g["g_zy"] * p["rho_zy"])[None, :] * rich.data["covariates"]
np.testing.assert_array_equal(expected_control, rich.data["covariate_contribution"])
print(
    "Reference targets reproduced; literal control contribution verified on float64 reported inputs."
)

active = corpus["treatment_active_mask"] == 1
np.testing.assert_allclose(
    corpus["beta"][active] * corpus["treatment_reference_response"][active],
    corpus["treatment_reference_contribution"][active],
    rtol=3e-14,
    atol=0.0,
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
axes[0].hist(np.log10(corpus["mm_kappa_mult"][active]), bins=8, edgecolor="white")
axes[0].set(
    xlabel="log10(kappa_mult)",
    ylabel="active (world, treatment) pairs",
    title="Realized MM scales — tiny sample, not a distribution test",
)
relative_input = np.linspace(0, 5, 200)
first_graph = layout.unpack(corpus["g"][0])
for slot in np.flatnonzero(first_graph["cy"]):
    response = (
        corpus["beta"][0, slot]
        * relative_input
        / (relative_input + corpus["mm_kappa_mult"][0, slot])
    )
    axes[1].plot(relative_input, response, label=f"C{slot + 1}")
axes[1].axvline(
    cfg.treatment_reference_multiplier, color="grey", linestyle="--", label="reference input"
)
axes[1].set(
    xlabel="post-carryover input / parameter-only anchor",
    ylabel="nominal contribution",
    title="Near-linear to saturated responses in world 0",
)
axes[1].legend(fontsize=7)
plt.show()
plt.close(fig)
rich_bundle = pg.write_scm_bundle(rich, str(RUN_DIR / "all-components"))
display(Image(filename=str(Path(rich_bundle) / "dag.png")))
display(Image(filename=str(Path(rich_bundle) / "decomposition.png")))
display(Image(filename=str(Path(rich_bundle) / "treatments.png")))

### Reusable template: evaluate the same world, not a second seed

The experimental template compiles at maximum sizes and receives graphs, activity masks, discrete families, and component flags as runtime data. It is **not** a switch on `sample_prior_predictive`, nor a complete accepted-corpus pipeline. Shocks, prior conditioning, and non-fixed confounding-strength ranges are outside its current scope.

Here it recomputes **every forward output of the already accepted world** using that world's exact primitive draws. No extra dataset is sampled. Matching a seed across the ordinary and padded template paths does not align their random variables; same-input evaluation is the relevant comparison.


In [ ]:
from pytensor.graph.replace import clone_replace

from pymc_generator.world_model_template import build_cell_inputs, build_world_model_template


def evaluate_at_primitives(model, names, primitives):
    replacements = {rv: rv.type(name=f"given_{rv.name}") for rv in model.free_RVs}
    expressions = clone_replace(
        [model[name] for name in names], replace=replacements, rebuild_strict=False
    )
    evaluate = pytensor.function(
        list(replacements.values()), expressions, on_unused_input="ignore", mode="FAST_COMPILE"
    )
    values = [np.asarray(primitives[rv.name], dtype=rv.dtype) for rv in replacements]
    return dict(zip(names, evaluate(*values)))


active_inputs = {
    "active_treatment": np.ones(rich.n_treatments),
    "active_covariate": np.ones(rich.n_covariates),
    "active_latent": np.ones(rich.n_latent),
}
cell = build_cell_inputs(rich.cfg, rich.g, active_inputs, rich.extras["structural"])
template, template_names, _ = build_world_model_template(rich.cfg, cell, rich.n_time_steps)
assert set(template_names) == set(rich.data)
template_values = evaluate_at_primitives(template, template_names, rich.primitive_parameters)
for name, expected in rich.data.items():
    if np.issubdtype(expected.dtype, np.integer):
        np.testing.assert_array_equal(template_values[name], expected, err_msg=name)
    else:
        np.testing.assert_allclose(
            template_values[name], expected, rtol=2e-11, atol=2e-11, err_msg=name
        )
print(
    "Template parity:",
    len(template_values),
    "forward outputs evaluated at the same accepted primitives",
)

### Fixed-input oracle: forward truth, without fitting a posterior

An oracle conditions on the **already scheduled observed inputs**; it must not apply the gates or envelopes a second time. We use `latent="sampled"` and bind the same primitive baseline/latent innovations, then compare its contribution and conditional outcome mean with generating truth.

With positive-decay geometric carryover and pre-window burn-in, the first `l_max - 1` reported rows need unpersisted treatment history. The oracle's reproducible likelihood window discards those rows. We compare only that window. Its conditional mean excludes observation noise. The default `latent="marginal"` instead integrates latent walks and cannot be compared with a realized baseline as though they were the same mean.

This demonstrates forward correctness, **not posterior recovery or convergence**. The separate [MMM recovery case study](https://pymc-labs.github.io/pymc-generator/examples/simple-model/) performs posterior fitting.


In [ ]:
oracle = rich.oracle_model(latent="sampled")
oracle_values = evaluate_at_primitives(
    oracle, ("contributions", "outcome_mu"), rich.primitive_parameters
)
warmup = (
    rich.cfg.l_max - 1 if rich.cfg.carryover_burn_in > 0 else 0
)  # this recipe uses positive-decay geometric kernels
np.testing.assert_allclose(
    oracle_values["contributions"][warmup:],
    rich.data["contributions_observed"][warmup:],
    rtol=2e-12,
    atol=2e-12,
)
np.testing.assert_allclose(
    oracle_values["outcome_mu"][warmup:],
    (rich.data["outcome"] - rich.data["outcome_noise"])[warmup:],
    rtol=2e-12,
    atol=2e-12,
)
print("Oracle forward values match on weeks", warmup, "through", rich.n_time_steps - 1)
print("Oracle free variables:", sorted(rv.name for rv in oracle.free_RVs))
print("No MCMC or additional dataset generation was used.")

## 9. Keep the datasets and decide what to change

The saved NPZ is the authoritative mixed shard with masks and truth. We also export one model-facing CSV per mixed world, omitting only padded inputs, and keep each atlas/replay world's human-readable bundle. `manifest.csv` records how to find the 17 datasets.

A practical loop:

1. Check schema and additive closure.
2. Read active-count coverage and **zero-weight exclusions**.
3. Check carried-component prevalence, measured visible shapes, and absent combinations.
4. Inspect rank/VIF in differences **and** levels, signal strength, and outcome shares.
5. Plot seeded examples; hand-pick unusual worlds by original ID.
6. Change the prior deliberately, generate another shard, and use `compare_descriptors` on a frozen reference.

Do not equate a missing trajectory in eight worlds with absent generator support. Increase **cells**, not merely draws per cell, to diversify graph sizes and component combinations. Turning features on can change other same-seed draws through graph/RNG layout; identical seeds across recipes are not controlled experiments. With features off, the library's existing golden checks guard its original laws and RNG streams. Inside one compiled template, runtime flags do not move primitive draws.

Only schema-v5 archives carry the new component truth; older archives are rejected, not silently zero-filled. Baseline seasonality has not been added, and seasonal inputs are not Fourier controls inserted into the baseline.


In [ ]:
manifest_rows = []
for world_id in range(len(corpus["cell_id"])):
    data = {"week": np.arange(corpus["outcome_raw"].shape[1])}
    for role, key, prefix in (
        ("treatment", "treatment_raw", "C"),
        ("covariate", "covariates", "Z"),
    ):
        for slot in np.flatnonzero(corpus[f"{role}_active_mask"][world_id]):
            data[f"{role}_{prefix}{slot + 1}"] = corpus[key][world_id, :, slot]
    data["outcome_Y"] = corpus["outcome_raw"][world_id]
    csv_path = RUN_DIR / f"mixed-world-{world_id}.csv"
    pd.DataFrame(data).to_csv(csv_path, index=False)
    manifest_rows.append(
        {
            "dataset": f"mixed:{world_id}",
            "source": "mixed shard",
            "world_id": world_id,
            "generation_seed": GENERATION_SEED,
            "model_data": str(csv_path.relative_to(RUN_DIR)),
        }
    )
for index, (name, bundle) in enumerate(atlas_paths.items()):
    manifest_rows.append(
        {
            "dataset": name,
            "source": "trajectory atlas",
            "world_id": 0,
            "generation_seed": 100 + index,
            "model_data": str((Path(bundle) / "dataset.csv").relative_to(RUN_DIR)),
        }
    )
manifest_rows.append(
    {
        "dataset": "all_components",
        "source": "replay world",
        "world_id": 0,
        "generation_seed": 27,
        "model_data": str((Path(rich_bundle) / "dataset.csv").relative_to(RUN_DIR)),
    }
)
manifest = pd.DataFrame(manifest_rows)
assert len(manifest) == 17
manifest.to_csv(RUN_DIR / "manifest.csv", index=False)
display(manifest)
print("Dataset and report directory:", RUN_DIR.resolve())
print("Stored mixed shard:", CORPUS_PATH.resolve())
print("Design screen:", report["decision"])
print(
    "Measured gaps:",
    report["inventory"]
    .loc[report["inventory"]["count"] == 0, ["role", "trajectory"]]
    .to_dict("records"),
)